In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATASET_PATH = "/content/drive/MyDrive/pr/Data Set"

In [3]:
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

In [4]:
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 814 files belonging to 5 classes.
Using 652 files for training.
Found 814 files belonging to 5 classes.
Using 162 files for validation.
Classes: ['hole', 'horizontal', 'lines', 'no defects', 'verticle']


In [5]:
normalization = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization(x), y))
val_ds = val_ds.map(lambda x, y: (normalization(x), y))

In [6]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [7]:
labels = np.concatenate([y for x, y in train_ds], axis=0)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(0.5821428571428572), 1: np.float64(1.2419047619047618), 2: np.float64(1.0866666666666667), 3: np.float64(1.0432), 4: np.float64(1.6717948717948719)}


In [8]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224,224,3)),

    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

In [9]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights
)

Epoch 1/15
21/21 ━━━━━━━━━━━━━━━━━━━━ 109s 5s/step - accuracy: 0.3804 - loss: 1.3808 - val_accuracy: 0.5185 - val_loss: 1.0053
Epoch 2/15
21/21 ━━━━━━━━━━━━━━━━━━━━ 83s 4s/step - accuracy: 0.6058 - loss: 0.8493 - val_accuracy: 0.7654 - val_loss: 0.7070
Epoch 3/15
21/21 ━━━━━━━━━━━━━━━━━━━━ 78s 4s/step - accuracy: 0.7699 - loss: 0.5884 - val_accuracy: 0.7778 - val_loss: 0.5615
Epoch 4/15
21/21 ━━━━━━━━━━━━━━━━━━━━ 76s 4s/step - accuracy: 0.7883 - loss: 0.4892 - val_accuracy: 0.8272 - val_loss: 0.4484
Epoch 5/15
21/21 ━━━━━━━━━━━━━━━━━━━━ 82s 4s/step - accuracy: 0.8160 - loss: 0.4021 - val_accuracy: 0.8642 - val_loss: 0.3526
Epoch 6/15
21/21 ━━━━━━━━━━━━━━━━━━━━ 78s 4s/step - accuracy: 0.8144 - loss: 0.3819 - val_accuracy: 0.8519 - val_loss: 0.3682
Epoch 7/15
21/21 ━━━━━━━━━━━━━━━━━━━━ 85s 4s/step - accuracy: 0.8206 - loss: 0.3592 - val_accuracy: 0.8642 - val_loss: 0.4091
Epoch 8/15
21/21 ━━━━━━━━━━━━━━━━━━━━ 87s 4s/step - accuracy: 0.8482 - loss: 0.3524 - val_accuracy: 0.8148 - val_loss

In [11]:
model.save("/content/drive/MyDrive/pr/defect_model.keras")

In [12]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
from google.colab import files

model = tf.keras.models.load_model("/content/drive/MyDrive/pr/defect_model.keras")

In [13]:
class_names = [
    "hole",
    "horizontal",
    "lines",
    "no defect",
    "vertical"
]

In [14]:
def preprocess(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img = image.img_to_array(img)
    img = img / 255.0
    return np.expand_dims(img, axis=0)

In [15]:
def predict():

    uploaded = files.upload()
    img_path = list(uploaded.keys())[0]

    img = preprocess(img_path)

    pred = model.predict(img)

    idx = np.argmax(pred)
    confidence = np.max(pred) * 100

    label = class_names[idx]

    print("\n===== RESULT =====")
    print("Prediction:", label)
    print("Confidence:", f"{confidence:.2f}%")

    if label == "no_defect" or confidence < 60:
        print("GOOD QUALITY (NO DEFECT)")
    else:
        print("DEFECTIVE ITEM")

In [19]:
predict()

Saving OIP (2).jpg to OIP (2) (1).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step

===== RESULT =====
Prediction: no defect
Confidence: 100.00%
DEFECTIVE ITEM
